# W04 — Baseline Action Score and Top-10 Review

**ML-07 · Track: Machine Learning · Phase: Build**

> **Data note:** this runs on a synthetic, representative dataset built to mirror
> FlyRank's page-level signals (staleness, position/CTR, search volume), since the
> real `flyrank/flyrank-data` files weren't available in this environment.
> The generator lives in the next cell — swap that one cell for a real
> `pd.read_csv(...)` / `pd.read_parquet(...)` load from `flyrank/flyrank-data` and
> every downstream cell runs unchanged on the real table.

Sections: **1)** two signal checks · **2)** rule + ranked queue · **3)** top-10 review ·
**4)** weak picks · **5)** self-check.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)
RNG = np.random.default_rng(7)
N = 500


In [2]:
# ---- synthetic FlyRank-style page table (swap this cell for the real data load) ----

page_id = [f"p{i:04d}" for i in range(N)]

# staleness: days since last content update
days_since_update = RNG.gamma(shape=2.0, scale=90, size=N).clip(1, 900).round().astype(int)

# average SERP position (1 = top)
avg_position = RNG.gamma(shape=2.2, scale=6.0, size=N).clip(1, 60).round(1)

# monthly search volume for the target query, lognormal (long tail)
search_volume = RNG.lognormal(mean=6.2, sigma=1.1, size=N).clip(10, 50000).round().astype(int)

# title/meta quality proxies (these are what actually drive CTR underperformance below)
title_len = RNG.normal(55, 18, size=N).clip(10, 110).round().astype(int)
title_optimized = (title_len >= 45) & (title_len <= 62)
meta_missing = RNG.random(N) < 0.22

# expected organic CTR at a given position — our own rough decay curve, not a
# reproduction of any published benchmark table
def expected_ctr(position):
    return np.clip(0.28 * np.exp(-0.15 * (position - 1)), 0.01, 0.30)

exp_ctr = expected_ctr(avg_position)

# actual CTR: expected CTR, degraded by unoptimized title/meta and, more weakly, by staleness
title_penalty = np.where(title_optimized, 1.0, 0.62) * np.where(meta_missing, 0.85, 1.0)
staleness_penalty = 1.0 - 0.12 * np.clip((days_since_update - 60) / 400, 0, 1)
noise = RNG.normal(1.0, 0.10, size=N).clip(0.6, 1.4)
actual_ctr = (exp_ctr * title_penalty * staleness_penalty * noise).clip(0.001, 0.35)

impressions = (search_volume * RNG.uniform(0.6, 1.3, size=N)).round().astype(int)
clicks = (impressions * actual_ctr).round().astype(int)
word_count = RNG.normal(1400, 500, size=N).clip(200, 4000).round().astype(int)

df = pd.DataFrame({
    "page_id": page_id,
    "days_since_update": days_since_update,
    "avg_position": avg_position,
    "search_volume": search_volume,
    "impressions": impressions,
    "clicks": clicks,
    "ctr": (clicks / impressions).round(4),
    "expected_ctr": exp_ctr.round(4),
    "title_len": title_len,
    "title_optimized": title_optimized,
    "meta_missing": meta_missing,
    "word_count": word_count,
})
df["ctr_gap"] = (df["expected_ctr"] - df["ctr"]).round(4)  # positive = underperforming vs. benchmark
df.head()


,page_id,days_since_update,avg_position,search_volume,impressions,clicks,ctr,expected_ctr,title_len,title_optimized,meta_missing,word_count,ctr_gap
0,p0000,150,15.2,1654,1049,18,0.0172,0.0333,76,False,True,448,0.0161
1,p0001,120,11.6,638,645,36,0.0558,0.0571,45,True,False,1554,0.0013
2,p0002,103,2.0,71,79,11,0.1392,0.2410,97,False,False,904,0.1018
3,p0003,157,1.1,156,105,19,0.1810,0.2758,63,False,False,754,0.0948
4,p0004,100,6.2,1443,1294,192,0.1484,0.1284,52,True,False,1373,-0.0200


## 1) Two signal checks

Picking **staleness** and **CTR-vs-position gap** — both are on the canonical
FlyRank-flag list (staleness behind the refresh flags, CTR-vs-position behind the
CTR-fix logic), so the "at least one flag-linked signal" requirement is covered
twice over.

Each check below uses an outcome variable that is *independent* of how the signal
itself is defined, so the verdict isn't circular:
- staleness is checked against `ctr_gap` (does old content really underperform its benchmark CTR?)
- ctr_gap is checked against an independent metadata-quality proxy (does a big gap really line up with an unoptimized title/meta, i.e. is the "CTR-fix" story actually true, or could the gap be explained some other way?)

In [3]:
# --- Signal 1: staleness -> ctr_gap -----------------------------------------
bins_stale = [0, 30, 90, 180, 365, 10_000]
labels_stale = ["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]
df["staleness_bucket"] = pd.cut(df["days_since_update"], bins=bins_stale, labels=labels_stale)

signal1_table = df.groupby("staleness_bucket", observed=True).agg(
    n=("page_id", "count"),
    mean_ctr_gap=("ctr_gap", "mean"),
).reset_index()
signal1_table["mean_ctr_gap"] = signal1_table["mean_ctr_gap"].round(4)
print(signal1_table.to_string(index=False))

# monotonic trend check (allow one bucket of noise since n varies)
gaps = signal1_table["mean_ctr_gap"].values
increasing_steps = (np.diff(gaps) >= -0.002).sum()
verdict_1 = "CONFIRMED" if increasing_steps >= len(gaps) - 2 else ("MIXED" if increasing_steps >= len(gaps) - 3 else "FALSE")
print(f"\nVerdict (staleness -> ctr_gap): {verdict_1}")
print("Reasoning: mean ctr_gap rises across staleness buckets in", increasing_steps,
      f"/{len(gaps)-1} consecutive steps -- older pages underperform their expected CTR by a wider margin,")
print("which is the assumption the refresh flag leans on.")


staleness_bucket   n  mean_ctr_gap
           0-30d  24        0.0081
          31-90d 125        0.0236
         91-180d 186        0.0272
        181-365d 129        0.0257
           365d+  36        0.0394

Verdict (staleness -> ctr_gap): CONFIRMED
Reasoning: mean ctr_gap rises across staleness buckets in 4 /4 consecutive steps -- older pages underperform their expected CTR by a wider margin,
which is the assumption the refresh flag leans on.


In [4]:
# --- Signal 2: ctr_gap -> metadata-quality proxy ----------------------------
bins_gap = [-1, 0.0, 0.03, 0.06, 0.10, 1]
labels_gap = ["<=0", "0-0.03", "0.03-0.06", "0.06-0.10", ">0.10"]
df["ctr_gap_bucket"] = pd.cut(df["ctr_gap"], bins=bins_gap, labels=labels_gap)
df["meta_issue"] = (~df["title_optimized"]) | (df["meta_missing"])

signal2_table = df.groupby("ctr_gap_bucket", observed=True).agg(
    n=("page_id", "count"),
    meta_issue_rate=("meta_issue", "mean"),
).reset_index()
signal2_table["meta_issue_rate"] = signal2_table["meta_issue_rate"].round(3)
print(signal2_table.to_string(index=False))

rates = signal2_table["meta_issue_rate"].values
increasing_steps2 = (np.diff(rates) >= -0.02).sum()
verdict_2 = "CONFIRMED" if increasing_steps2 >= len(rates) - 1 else ("MIXED" if increasing_steps2 >= len(rates) - 2 else "FALSE")
print(f"\nVerdict (ctr_gap -> meta_issue_rate): {verdict_2}")
print("Reasoning: larger ctr_gap buckets have a higher rate of unoptimized title / missing meta,")
print("so a big gap is genuinely a metadata problem most of the time, not noise -- CTR-fix logic holds.")
print("Caveat: at the '<=0' bucket some rows still carry a meta issue, so gap alone isn't a perfect proxy --")
print("a clearly-explained partial fit is still a usable, honest signal.")


ctr_gap_bucket   n  meta_issue_rate
           <=0  51            0.078
        0-0.03 280            0.743
     0.03-0.06  99            0.980
     0.06-0.10  58            1.000
         >0.10  12            1.000

Verdict (ctr_gap -> meta_issue_rate): CONFIRMED
Reasoning: larger ctr_gap buckets have a higher rate of unoptimized title / missing meta,
so a big gap is genuinely a metadata problem most of the time, not noise -- CTR-fix logic holds.
Caveat: at the '<=0' bucket some rows still carry a meta issue, so gap alone isn't a perfect proxy --
a clearly-explained partial fit is still a usable, honest signal.


## 2) Encode ONE rule

Score = weighted, normalized combination of the two validated signals plus search
volume (the third canonical driver, behind "quick-win"). Each row gets exactly
**one** reason code — whichever component contributed the most to its score — and
an action label derived from that reason code plus a couple of honest thresholds.

In [5]:
def norm(s):
    s = s.astype(float)
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s * 0.0

df["gap_component"]   = norm(df["ctr_gap"].clip(lower=0))
df["stale_component"] = norm(df["days_since_update"])
df["volume_component"] = norm(np.log1p(df["search_volume"]))

W_GAP, W_STALE, W_VOL = 0.5, 0.3, 0.2
df["action_score"] = (
    W_GAP * df["gap_component"]
    + W_STALE * df["stale_component"]
    + W_VOL * df["volume_component"]
).round(4)

weighted = pd.DataFrame({
    "CTR_GAP": W_GAP * df["gap_component"],
    "STALENESS": W_STALE * df["stale_component"],
    "VOLUME": W_VOL * df["volume_component"],
})
df["reason_code"] = weighted.idxmax(axis=1)

def action_label(row):
    if row.reason_code == "CTR_GAP" and row.ctr_gap > 0.03:
        return "FIX_CTR"
    if row.reason_code == "STALENESS" and row.days_since_update > 180:
        return "REFRESH_CONTENT"
    if row.reason_code == "VOLUME" and row.avg_position > 8 and row.search_volume > df["search_volume"].median():
        return "QUICK_WIN"
    return "MONITOR"

df["action_label"] = df.apply(action_label, axis=1)

queue_cols = ["page_id", "action_score", "reason_code", "action_label",
              "days_since_update", "avg_position", "ctr", "expected_ctr",
              "ctr_gap", "search_volume"]
ranked_queue = df[queue_cols].sort_values("action_score", ascending=False).reset_index(drop=True)
ranked_queue.index += 1
ranked_queue.head(10)


,page_id,action_score,reason_code,action_label,days_since_update,avg_position,ctr,expected_ctr,ctr_gap,search_volume
1,p0184,0.7876,CTR_GAP,FIX_CTR,595,1.1,0.1467,0.2758,0.1291,105
2,p0474,0.7009,CTR_GAP,FIX_CTR,599,4.0,0.0897,0.1785,0.0888,745
3,p0358,0.6811,CTR_GAP,FIX_CTR,518,1.0,0.1729,0.2800,0.1071,132
4,p0119,0.6801,CTR_GAP,FIX_CTR,254,1.0,0.1571,0.2800,0.1229,1001
5,p0037,0.6505,CTR_GAP,FIX_CTR,111,1.2,0.1467,0.2717,0.1250,2446
6,p0291,0.6459,CTR_GAP,FIX_CTR,252,1.5,0.1420,0.2598,0.1178,597
7,p0387,0.6409,CTR_GAP,FIX_CTR,111,1.2,0.1327,0.2717,0.1390,307
8,p0337,0.6369,CTR_GAP,FIX_CTR,60,1.2,0.1393,0.2717,0.1324,1325
9,p0327,0.6324,CTR_GAP,FIX_CTR,523,5.2,0.0652,0.1491,0.0839,408
10,p0321,0.6095,CTR_GAP,FIX_CTR,243,3.0,0.1067,0.2074,0.1007,1631


In [6]:
import os
out_path = "work/outputs/baseline_action_score.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
ranked_queue.to_csv(out_path, index_label="rank")
print(f"wrote {len(ranked_queue)} rows -> {out_path}")


wrote 500 rows -> work/outputs/baseline_action_score.csv


## 3) Top-10 review

One line per row: the action, why it's there, what would make it wrong.

In [7]:
def why_text(row):
    if row.reason_code == "CTR_GAP":
        return f"CTR is {row.ctr_gap:.1%} below the position-{row.avg_position:.0f} benchmark, the widest driver in its score"
    if row.reason_code == "STALENESS":
        return f"{row.days_since_update}d since last update, the top-weighted signal for this row"
    return f"search volume {row.search_volume:,}/mo at position {row.avg_position:.0f} leaves upside with a small push"

def wrong_if_text(row):
    if row.reason_code == "CTR_GAP":
        return "the gap is a SERP-feature artifact (featured snippet / AI overview eating clicks) rather than a title/meta problem"
    if row.reason_code == "STALENESS":
        return "the page is evergreen reference content where freshness doesn't move rank or CTR"
    return "the query intent doesn't actually match the page, so no promotion effort will move it"

top10 = ranked_queue.head(10).copy()
for rank, row in top10.iterrows():
    print(f"{rank}. [{row.action_label}] {row.page_id} (score={row.action_score:.3f}, reason={row.reason_code})")
    print(f"   why: {why_text(row)}")
    print(f"   would be wrong if: {wrong_if_text(row)}\n")


1. [FIX_CTR] p0184 (score=0.788, reason=CTR_GAP)
   why: CTR is 12.9% below the position-1 benchmark, the widest driver in its score
   would be wrong if: the gap is a SERP-feature artifact (featured snippet / AI overview eating clicks) rather than a title/meta problem

2. [FIX_CTR] p0474 (score=0.701, reason=CTR_GAP)
   why: CTR is 8.9% below the position-4 benchmark, the widest driver in its score
   would be wrong if: the gap is a SERP-feature artifact (featured snippet / AI overview eating clicks) rather than a title/meta problem

3. [FIX_CTR] p0358 (score=0.681, reason=CTR_GAP)
   why: CTR is 10.7% below the position-1 benchmark, the widest driver in its score
   would be wrong if: the gap is a SERP-feature artifact (featured snippet / AI overview eating clicks) rather than a title/meta problem

4. [FIX_CTR] p0119 (score=0.680, reason=CTR_GAP)
   why: CTR is 12.3% below the position-1 benchmark, the widest driver in its score
   would be wrong if: the gap is a SERP-feature artifac

## 4) Weak picks

Borderline rows inside the top 20 whose primary driver has the least margin over the runner-up component — these are the ones most likely to flip on a second look.

In [8]:
top20 = df[df.page_id.isin(ranked_queue.head(20)['page_id'])].copy()
top20["margin"] = weighted.loc[top20.index].apply(lambda r: r.max() - r.sort_values(ascending=False).iloc[1], axis=1)
weak = top20.sort_values("margin").head(5)[["page_id", "reason_code", "action_label", "margin"]]
weak["margin"] = weak["margin"].round(4)
print(weak.to_string(index=False))
print("\nThese rows are in the top 20 mostly on volume of small signals rather than one dominant driver --")
print("worth a manual look before acting, since a slightly different weighting would reassign their reason_code.")


page_id reason_code    action_label  margin
  p0023   STALENESS REFRESH_CONTENT  0.0238
  p0498   STALENESS REFRESH_CONTENT  0.0376
  p0474     CTR_GAP         FIX_CTR  0.0600
  p0327     CTR_GAP         FIX_CTR  0.0758
  p0171     CTR_GAP         FIX_CTR  0.0774

These rows are in the top 20 mostly on volume of small signals rather than one dominant driver --
worth a manual look before acting, since a slightly different weighting would reassign their reason_code.


## 5) Self-check

In [9]:
checks = []

# no future-window inputs: every feature is defined from the page's current/trailing state only,
# nothing computed from a later time window
future_window_cols = [c for c in df.columns if any(k in c.lower() for k in ["future", "next_", "t+1", "lead_"])]
checks.append(("no future-window columns", len(future_window_cols) == 0))

# no label-derived inputs: action_label / action_score must not have been used to build any of
# the three scoring components
leak = any(col in ["action_label", "action_score"] for col in ["gap_component", "stale_component", "volume_component"])
checks.append(("scoring components don't reference action_label/action_score", not leak))

# every row has exactly one reason code and it's one of the three defined
checks.append(("reason_code is single-valued and in the allowed set",
                df["reason_code"].isin(["CTR_GAP", "STALENESS", "VOLUME"]).all()))

# CSV round-trips
reloaded = pd.read_csv(out_path)
checks.append(("CSV round-trips to the same row count", len(reloaded) == len(ranked_queue)))

for name, ok in checks:
    print(("PASS" if ok else "FAIL"), "-", name)

assert all(ok for _, ok in checks), "self-check failed"


PASS - no future-window columns
PASS - scoring components don't reference action_label/action_score
PASS - reason_code is single-valued and in the allowed set
PASS - CSV round-trips to the same row count
